<a href="https://colab.research.google.com/github/bzkzhao/bzk-omics/blob/main/colab_identityresolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week C — resolve identities

**Corrected version.** The first run stripped isoform suffixes (`P09914-2` → `P09914`) and validated positions against the *canonical* sequence. Two of twenty sites failed as a result — not because the data was wrong, but because the resolver compared against the wrong sequence.

That bug exposed a schema defect: isoform must be part of the `ModificationSite` key, not a property of it. `ONTOLOGY.md` §4 is now corrected.

**Goal:** test invariant I2 against real data — does the position MaxQuant reported actually hold a lysine in the sequence it was computed against?

**This code is not throwaway.** Steps 4 and 5 are the resolver module in `bzk/resolve/`, minus caching.

## Step 1 — load the site table

In [ ]:
import pandas as pd, numpy as np, requests, os, time, json

URL = ("https://ftp.pride.ebi.ac.uk/pride/data/archive/2022/02/"
       "PXD018299/HAP1_USP18KO_GlyGlyKSites.txt")
LOCAL = "HAP1_USP18KO_GlyGlyKSites.txt"

if not os.path.exists(LOCAL):
    with requests.get(URL, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(LOCAL, "wb") as fh:
            for chunk in r.iter_content(chunk_size=1 << 20):
                fh.write(chunk)

df = pd.read_csv(LOCAL, sep="\t", low_memory=False)
clean = df[(df["Reverse"] != "+") & (df["Potential contaminant"] != "+")].copy()
print(f"{len(clean):,} sites after filtering")

2,298 sites after filtering


## Step 2 — what the file claims

`Protein` is the razor pick, `Proteins` the full candidate set, `Position` the residue number.

There is **no sequence version anywhere in this file**. That is the gap I2 identifies: the position is stated against a sequence the file does not identify.

In [ ]:
cols = [c for c in ["Protein", "Proteins", "Gene names", "Position",
                    "Amino acid", "Sequence window", "Localization prob"]
        if c in clean.columns]

n_iso = clean["Protein"].astype(str).str.contains("-").sum()
print(f"razor picks that are isoforms: {n_iso:,} of {len(clean):,} "
      f"({100*n_iso/len(clean):.0f}%)\n")
clean[cols].head(10)

razor picks that are isoforms: 334 of 2,298 (15%)



,Protein,Proteins,Gene names,Position,Amino acid,Sequence window,Localization prob
0,A0A024R4E5,A0A024R4E5;Q00341-2;Q00341;C9JHS7;C9JHZ8;C9JES...,HDLBP,88.0,K,ASVITQVFHVPLEERKYKDMNQFGEGEQAKI,0.499996
1,A0A024R4E5,A0A024R4E5;Q00341-2;Q00341;C9JHS7;C9JHZ8;C9JES...,HDLBP,90.0,K,VITQVFHVPLEERKYKDMNQFGEGEQAKICL,0.868171
2,A0A024R4E5,A0A024R4E5;Q00341-2;Q00341;H0Y394;H7BZC3;H7C2D1,HDLBP,672.0,K,EVSIPAKLHNSLIGTKGRLIRSIMEECGGVH,1.000000
3,A0A024R4E5,A0A024R4E5;Q00341-2;Q00341;H0Y394;H7BZC3,HDLBP,599.0,K,ISVPIFKQFHKNIIGKGGANIKKIREESNTK,0.749654
4,A0A024R4E5,A0A024R4E5;Q00341-2;Q00341;H0Y394;H7BZC3,HDLBP,605.0,K,KQFHKNIIGKGGANIKKIREESNTKIDLPAE,0.576795
5,A0A024R4M0,A0A024R4M0;P46781,RPS9,155.0,K,QVVNIPSFIVRLDSQKHIDFSLRSPYGGGRP,1.000000
6,A0A024R4M0,A0A024R4M0;P46781;B5MCT8;C9JM19,RPS9,121.0,K,LERRLQTQVFKLGLAKSIHHARVLIRQRHIR,1.000000
7,A0A024R571,A0A024R571;Q9H4M9;Q9NZN3;C9JC03;C9JIJ3;C9IZH1;...,EHD1;EHD3,138.0,K,VPGNALVVDPRRPFRKLNAFGNAFLNRFMCA,1.000000
8,A0A024RA52,A0A024RA52;P25787,PSMA2,50.0,K,SVGIKAANGVVLATEKKQKSILYDERSVHKV,0.500474
9,A0A024RA52,A0A024RA52;P25787;C9JCK5,PSMA2,165.0,K,DPSGAYFAWKATAMGKNYVNGKTFLEKRYNE,0.999020


## Step 3 — take a sample

In [ ]:
N = 20
sample = (clean[clean["Localization prob"] >= 0.9]
          .sample(n=N, random_state=0)[cols]
          .reset_index(drop=True))
sample

,Protein,Proteins,Gene names,Position,Amino acid,Sequence window,Localization prob
0,A0A087WXQ8,A0A087WXQ8;A0A087WY10;A0A087WUL2;P49720,PSMB3,77.0,K,TVAQRLKFRLNLYELKEGRQIKPYTLMSMVA,1.000000
1,P29401,P29401;P29401-2;A0A0B4J1R6;P51854-4;P51854-1;P...,TKT;TKTL1,314.0,K,ANIRMPSLPSYKVGDKIATRKAYGQALAKLG;TDVRMTSPPDYRVG...,1.000000
2,H0YKK0,H0YKK0;P09661;H0YMA0,SNRPA1,149.0,K,YVIYKVPQVRVLDFQKVKLKF__________,0.924368
3,J3KTA4,J3KTA4;P17844;P17844-2,DDX5,388.0,K,GDKSQQERDWVLNEFKHGKAPILIATDVASR,0.965845
4,P52272-2,P52272-2;P52272;A0A087X0X3;M0R019;M0R2T0;M0QYQ...,HNRNPM,145.0,K,EVLNKHSLSGRPLKVKEDPDGEHARRAMQKA,1.000000
5,P09874,P09874,PARP1,400.0,K,SADKPLSNMKILTLGKLSRNKDEVKAMIEKL,1.000000
6,F8VNX8,F8VNX8;O14545;F8VVF3;O14545-2,TRAFD1,103.0,K,AVCQHCDLELSILKLKEHEDYCGARTELCGN,1.000000
7,Q68EM7-2,Q68EM7-2;Q68EM7-6;Q68EM7-5;Q68EM7;I3L4P0;I3L1S...,ARHGAP17,48.0,K,ERRLDTVRSICHHSHKRLVACFQGQHGTDAE,1.000000
8,P60842,P60842;J3KT12;P60842-2;J3QS69;J3KTB5;J3QL43;J3...,EIF4A1,146.0,K,CHACIGGTNVRAEVQKLQMEAPHIIVGTPGR,1.000000
9,P22059,P22059,OSBP,292.0,K,RDFLMLAQTHSKKWQKSLQYERDQRIRLEET,1.000000


## Step 4 — ask UniProt, isoform-aware

**The fix.** The JSON endpoint only serves canonical entries, so isoforms need the FASTA endpoint at their full accession:

- `rest.uniprot.org/uniprotkb/P09914.json` → canonical IFIT1, with metadata
- `rest.uniprot.org/uniprotkb/P09914-2.fasta` → isoform 2 sequence

So this fetches the canonical JSON for metadata (sequence version, review status, gene) and, where an isoform is requested, replaces the sequence with the isoform's. Stripping the suffix returns a protein of different length whose positions are wrong **without erroring** — the worst kind of failure.

In [ ]:
def fetch_uniprot(accession):
    """Resolve an accession, honouring isoform suffixes."""
    acc = str(accession).strip()
    is_isoform = "-" in acc
    canonical = acc.split("-")[0]

    try:
        r = requests.get(f"https://rest.uniprot.org/uniprotkb/{canonical}.json",
                         timeout=30)
    except Exception as e:
        return {"status": "network_error", "detail": str(e)}

    if r.status_code == 404:
        return {"status": "not_found"}
    if r.status_code != 200:
        return {"status": f"http_{r.status_code}"}

    d = r.json()
    out = {
        "status": "ok",
        "requested": acc,
        "canonical": canonical,
        "is_isoform": is_isoform,
        "entry_type": d.get("entryType", ""),
        "reviewed": "reviewed" in d.get("entryType", "").lower()
                    and "unreviewed" not in d.get("entryType", "").lower(),
        "sequence": d.get("sequence", {}).get("value", ""),
        "sequence_version": d.get("entryAudit", {}).get("sequenceVersion"),
        "last_seq_update": d.get("entryAudit", {}).get("lastSequenceUpdateDate"),
        "gene": (d.get("genes") or [{}])[0].get("geneName", {}).get("value"),
        "sequence_source": "canonical",
    }

    if is_isoform:
        try:
            f = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.fasta",
                             timeout=30)
            if f.status_code == 200 and f.text.startswith(">"):
                seq = "".join(f.text.split("\n")[1:]).strip()
                if seq:
                    out["sequence"] = seq
                    out["sequence_source"] = "isoform"
            else:
                out["sequence_source"] = "isoform_unavailable"
        except Exception:
            out["sequence_source"] = "isoform_fetch_failed"

    return out


# sanity check: the two sites that failed in the first run
for acc, pos in [("P09914", 376), ("P09914-2", 376),
                 ("P62195", 47), ("P62195-2", 47)]:
    info = fetch_uniprot(acc)
    time.sleep(0.3)
    seq = info.get("sequence", "")
    aa = seq[pos - 1] if len(seq) >= pos else "-"
    print(f"{acc:12s} pos {pos:4d}  len={len(seq):5d}  "
          f"residue={aa}  source={info.get('sequence_source')}")

P09914       pos  376  len=  478  residue=T  source=canonical
P09914-2     pos  376  len=  447  residue=K  source=isoform
P62195       pos   47  len=  406  residue=A  source=canonical
P62195-2     pos   47  len=  398  residue=K  source=isoform


The canonical rows should return T and A; the isoform rows should return K. Same position, same gene, different sequence — which is the whole argument for putting isoform in the key.

## Step 5 — validate each site

| Outcome | Meaning | Schema |
|---|---|---|
| `ok` | Position holds a K | `ModificationSite` created |
| `wrong_residue` | Position holds something else | Genuine drift — I2 firing |
| `out_of_range` | Position beyond sequence end | Wrong sequence or wrong accession |
| `isoform_unavailable` | Isoform sequence not retrievable | Cannot validate; must not assume canonical |
| `not_found` | Accession retired or merged | Needs the ID-mapping service |

In [ ]:
results = []

for _, row in sample.iterrows():
    acc = str(row["Protein"])
    pos = int(row["Position"]) if not pd.isna(row["Position"]) else None
    info = fetch_uniprot(acc)
    time.sleep(0.4)

    rec = {"accession": acc, "position": pos,
           "is_isoform": "-" in acc,
           "gene_in_file": row.get("Gene names"),
           "status": info["status"]}

    if info["status"] != "ok":
        rec["check"] = info["status"]
    elif info["sequence_source"].startswith("isoform_"):
        rec["check"] = "isoform_unavailable"
    else:
        seq = info["sequence"]
        rec.update({
            "seq_version": info["sequence_version"],
            "last_seq_update": info["last_seq_update"],
            "reviewed": info["reviewed"],
            "seq_length": len(seq),
            "seq_source": info["sequence_source"],
        })
        if pos is None or pos > len(seq):
            rec["check"], rec["residue"] = "out_of_range", None
        else:
            aa = seq[pos - 1]
            rec["residue"] = aa
            rec["check"] = "ok" if aa == "K" else "wrong_residue"

    results.append(rec)
    print(f"{acc:14s} pos {str(pos):>5s}  {rec['check']}")

res = pd.DataFrame(results)
print("\n--- summary ---")
print(res["check"].value_counts())

A0A087WXQ8     pos    77  ok
P29401         pos   314  ok
H0YKK0         pos   149  ok
J3KTA4         pos   388  ok
P52272-2       pos   145  ok
P09874         pos   400  ok
F8VNX8         pos   103  ok
Q68EM7-2       pos    48  ok
P60842         pos   146  ok
P22059         pos   292  ok
Q9NVJ2         pos   141  ok
P62805         pos    32  ok
H7BZW7         pos    37  ok
P09914-2       pos   376  ok
Q8TC12-3       pos   105  ok
O60832-2       pos    61  ok
P22102         pos   251  ok
P62195-2       pos    47  ok
P18124         pos    59  ok
P25789         pos   231  ok

--- summary ---
check
ok    20
Name: count, dtype: int64


## Step 6 — the numbers that matter

In [ ]:
n = len(res)
ok = (res["check"] == "ok").sum()
print(f"validated:        {ok}/{n} ({100*ok/n:.0f}%)")
print(f"isoform sites:    {res['is_isoform'].sum()}/{n}")

if "reviewed" in res:
    print(f"reviewed entries: {res['reviewed'].sum()}/{n}")
if "seq_version" in res:
    print(f"sequence versions: "
          f"{sorted(res['seq_version'].dropna().unique().tolist())}")

bad = res[res["check"] != "ok"]
print("\nfailures:")
print(bad.to_string() if len(bad) else "none — isoform handling resolved them")

validated:        20/20 (100%)
isoform sites:    6/20
reviewed entries: 15/20
sequence versions: [1, 2, 3, 4]

failures:
none — isoform handling resolved them


## Step 7 — position drift

The site table records no sequence version, so positions were computed against a ~2019 FASTA. Where UniProt has amended a sequence since, a position that still validates as K may be a **different lysine** than the one measured.

That failure is completely silent. It is why I2 puts the version in the primary key.

In [ ]:
SEARCH_DATE = "2019-01-01"

if "last_seq_update" in res:
    upd = res.dropna(subset=["last_seq_update"]).copy()
    upd["amended"] = upd["last_seq_update"] > SEARCH_DATE
    n_amended = int(upd["amended"].sum())
    rate = n_amended / len(upd)
    print(f"amended since ~{SEARCH_DATE}: {n_amended} of {len(upd)} ({100*rate:.0f}%)")
    print(f"extrapolated across {len(clean):,} sites: ~{int(rate*len(clean)):,} at risk")
    if n_amended:
        print("\npositions that may refer to different residues than intended:")
        print(upd[upd["amended"]][["accession", "position", "seq_version",
                                   "last_seq_update", "check"]].to_string())

amended since ~2019-01-01: 1 of 20 (5%)
extrapolated across 2,298 sites: ~114 at risk

positions that may refer to different residues than intended:
   accession  position  seq_version last_seq_update check
12    H7BZW7        37            3      2026-06-10    ok


## Step 8 — reviewed versus unreviewed picks

**Also fixed.** The first version stripped isoform suffixes here too, so `P52272-2` reported as reviewed via its canonical entry. Isoform accessions inherit their canonical entry's review status, which is now stated explicitly rather than obscured.

The question: how often does the razor pick land on a TrEMBL accession when a reviewed Swiss-Prot entry sits in the same candidate set?

In [ ]:
_cache = {}

def review_status(acc):
    canonical = str(acc).split("-")[0]
    if canonical not in _cache:
        info = fetch_uniprot(canonical)
        time.sleep(0.4)
        _cache[canonical] = bool(info.get("reviewed"))
    return _cache[canonical]

rows = []
for _, row in sample.head(8).iterrows():
    picked = str(row["Protein"])
    candidates = [c.strip() for c in str(row["Proteins"]).split(";") if c.strip()][:6]
    reviewed = [c for c in candidates if review_status(c)]
    picked_reviewed = review_status(picked)
    missed = bool(reviewed) and not picked_reviewed
    rows.append({"picked": picked, "picked_reviewed": picked_reviewed,
                 "n_candidates": len(candidates),
                 "reviewed_available": reviewed[:3], "missed_opportunity": missed})
    print(f"{picked:14s} reviewed={str(picked_reviewed):5s} "
          f"of {len(candidates)} candidates"
          f"{'   <- reviewed alternative available' if missed else ''}")

swiss = pd.DataFrame(rows)
print(f"\nrazor pick unreviewed despite a reviewed alternative: "
      f"{swiss['missed_opportunity'].sum()} of {len(swiss)}")

A0A087WXQ8     reviewed=False of 4 candidates   <- reviewed alternative available
P29401         reviewed=True  of 6 candidates
H0YKK0         reviewed=False of 3 candidates   <- reviewed alternative available
J3KTA4         reviewed=False of 3 candidates   <- reviewed alternative available
P52272-2       reviewed=True  of 6 candidates
P09874         reviewed=True  of 1 candidates
F8VNX8         reviewed=False of 4 candidates   <- reviewed alternative available
Q68EM7-2       reviewed=True  of 6 candidates

razor pick unreviewed despite a reviewed alternative: 4 of 8


## Step 9 — the resolution record

What resolution found, in a form the graph can hold.

In [ ]:
record = {
    "dataset": "PXD018299",
    "file": LOCAL,
    "n_sampled": int(n),
    "n_validated": int(ok),
    "validation_rate": round(ok / n, 3),
    "n_isoform_sites": int(res["is_isoform"].sum()),
    "n_reviewed": int(res["reviewed"].sum()) if "reviewed" in res else None,
    "sequence_versions": sorted(res["seq_version"].dropna().unique().tolist())
                          if "seq_version" in res else [],
    "razor_pick_unreviewed_despite_alternative":
        int(swiss["missed_opportunity"].sum()),
    "razor_pick_sampled": int(len(swiss)),
    "resolver_notes": [
        "Isoform accessions resolved via the FASTA endpoint at full accession; "
        "never stripped to canonical.",
        "Review status is inherited from the canonical entry; isoforms are not "
        "independently reviewed.",
    ],
}

with open("resolution_PXD018299.json", "w") as fh:
    json.dump(record, fh, indent=2)
print(json.dumps(record, indent=2))
print("\nDownload alongside the curation and analysis records.")

{
  "dataset": "PXD018299",
  "file": "HAP1_USP18KO_GlyGlyKSites.txt",
  "n_sampled": 20,
  "n_validated": 20,
  "validation_rate": 1.0,
  "n_isoform_sites": 6,
  "n_reviewed": 15,
  "sequence_versions": [
    1,
    2,
    3,
    4
  ],
  "razor_pick_unreviewed_despite_alternative": 4,
  "razor_pick_sampled": 8,
  "resolver_notes": [
    "Isoform accessions resolved via the FASTA endpoint at full accession; never stripped to canonical.",
    "Review status is inherited from the canonical entry; isoforms are not independently reviewed."
  ]
}

Download alongside the curation and analysis records.


## Step 10 — what this settled

1. **Isoform belongs in the key, not as a property.** Found by a resolver bug, corrected in `ONTOLOGY.md` §4 and invariant I2.
2. **Reviewed entries must be preferred, and the preference recorded** — invariant I17.
3. **Position drift is real but rare** (~5%), so flagging is sufficient; historical sequence retrieval is not needed for v0.1. That resolves open question 1 in `ARCHITECTURE.md`.

**Next:** port Steps 4 and 5 into `bzk/resolve/uniprot.py` with a persistent cache, and write the invariant tests before the adapter.